## Adaptive R Matrix

In [1]:
import os, sys, glob, re
import numpy as np
from pathlib import Path
project_root_path = Path.cwd().resolve()
while not (project_root_path / "Backend").exists() and project_root_path.parent != project_root_path:
    project_root_path = project_root_path.parent
sys.path.insert(0, str(project_root_path))
os.chdir(project_root_path)
import RF_Track as rft
import numpy as np
import matplotlib.pyplot as plt
from Backend.ResponseMatrix_DFS_WFS import ResponseMatrix_DFS_WFS
from ComputeResponseMatrix_GUI import MainWindow
from Backend.State import State
CRM_GUI = MainWindow()
RM = ResponseMatrix_DFS_WFS()


RF-Track, version 2.6.3

Copyright (C) 2016-2026 CERN, Geneva, Switzerland. All rights reserved.

Author and contact:
 Andrea Latina <andrea.latina@cern.ch>
 BE-ABP Group
 CERN
 CH-1211 GENEVA 23
 SWITZERLAND

This software is distributed under a CERN proprietary software
license in the hope that it will be useful, but WITHOUT ANY WARRANTY;
not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

See the COPYRIGHT and LICENSE files at the top-level directory of
the RF-Track download area: https://gitlab.cern.ch/rf-track

RF-Track was compiled with GSL-2.8 and fftw-3.3.11



[RF-Track] Could not check for updates.


In [2]:
def _load_response(directory):
    folder = CRM_GUI._expand_path(directory)
    CRM_GUI._load_lists_from_directory(folder)
    response = CRM_GUI._compute_response_of_one_data_directory(directory=folder)
    matrix = _response_matrix(response)
    return response, matrix

In [3]:
def _response_matrix(response):
    R = np.block([
        [response.Rxx, response.Rxy],
        [response.Ryx, response.Ryy]
    ])
    return R

In [4]:
def list_pair_files(directory):
    datafiles = sorted(glob.glob(os.path.join(directory, "DATA*.pkl")))
    pair_re = re.compile(r"DATA_(.+)_(p|m)(\d+)\.pkl$")
    pairs = []
    for fp in datafiles:
        match = pair_re.search(os.path.basename(fp))
        if not match or match.group(2) != "p":
            continue
        tag = match.group(1)
        idx = match.group(3)
        fm = os.path.join(directory, f"DATA_{tag}_m{idx}.pkl")
        if os.path.exists(fm):
            pairs.append((fp, fm, tag))
    return pairs

In [5]:
def list_plus_files(directory):
    datafiles = sorted(glob.glob(os.path.join(directory, "DATA*.pkl")))
    plus_re = re.compile(r"DATA_(.+)_p(\d+)\.pkl$")
    plus_files = []
    for fp in datafiles:
        match = plus_re.search(os.path.basename(fp))
        if match:
            plus_files.append(fp)
    return plus_files

### Perfectly aligned machine

In [6]:
directory_aligned_orbit = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/AlignedMachine/ATF2_Ext_RFT_evr_aligned_Orbit"
directory_aligned_energy = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/AlignedMachine/ATF2_Ext_RFT_evr_aligned_Dispersion"
directory_aligned_charge = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/AlignedMachine/ATF2_Ext_RFT_evr_aligned_Wakefield"

In [7]:
R0_response_aligned, R0_aligned = _load_response(directory_aligned_orbit)
R1_response_aligned, R1_aligned = _load_response(directory_aligned_energy)
R2_response_aligned, R2_aligned = _load_response(directory_aligned_charge)

**Dispersion dataset:** response matrix measured after applying the DFS energy shift.
It represents the machine response at shifted beam energy.

How much the matrices actually differ from each other, with respect to R0?

We calculate the Frobenius norms:

In [8]:
norm_energy = np.linalg.norm(R1_aligned-R0_aligned, "fro") / np.linalg.norm(R0_aligned, "fro")
norm_charge = np.linalg.norm(R2_aligned-R0_aligned, "fro") / np.linalg.norm(R0_aligned, "fro")

print("Energy [%]: ", norm_energy * 100)
print("Charge [%]: ", norm_charge * 100)

Energy [%]:  82.23689738484683
Charge [%]:  1.024090825169679e-05


### BPMs misaligned by sigma 0.06mm rms.

In [9]:
directory_bpms_misaligned_006_orbit = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/BPMs_misaligned_0_06/ATF2_Ext_RFT_mis_bpms_0_06_Orbit"
directory_bpms_misaligned_006_energy = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/BPMs_misaligned_0_06/ATF2_Ext_RFT_mis_bpms_0_06_Dispersion"
directory_bpms_misaligned_006_charge = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/BPMs_misaligned_0_06/ATF2_Ext_RFT_mis_bpms_0_06_Wakefield"

In [10]:
R0_response_bpms6, R0_bpms6 = _load_response(directory_bpms_misaligned_006_orbit)
R1_response_bpms6, R1_bpms6 = _load_response(directory_bpms_misaligned_006_energy)
R2_response_bpms6, R2_bpms6 = _load_response(directory_bpms_misaligned_006_charge)

In [11]:
norm_energy = np.linalg.norm(R1_bpms6-R0_bpms6, "fro") / np.linalg.norm(R0_bpms6, "fro")
norm_charge = np.linalg.norm(R2_bpms6-R0_bpms6, "fro") / np.linalg.norm(R0_bpms6, "fro")

print("Energy [%]: ", norm_energy * 100)
print("Charge [%]: ", norm_charge * 100)

Energy [%]:  82.23673131758629
Charge [%]:  1.0240912389323866e-05


### BPMs misaligned by sigma 0.100 mm rms.

In [12]:
directory_bpms_misaligned_0100_orbit = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/BPMs_misaligned_0_100/ATF2_Ext_RFT_mis_bpms_0_100_Orbit"
directory_bpms_misaligned_0100_energy = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/BPMs_misaligned_0_100/ATF2_Ext_RFT_mis_bpms_0_100_Dispersion"
directory_bpms_misaligned_0100_charge = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/BPMs_misaligned_0_100/ATF2_Ext_RFT_mis_bpms_0_100_Wakefield"

In [13]:
R0_response_bpms100, R0_bpms100 = _load_response(directory_bpms_misaligned_0100_orbit)
R1_response_bpms100, R1_bpms100 = _load_response(directory_bpms_misaligned_0100_energy)
R2_response_bpms100, R2_bpms100 = _load_response(directory_bpms_misaligned_0100_charge)

In [14]:
norm_energy = np.linalg.norm(R1_bpms100-R0_bpms100, "fro") / np.linalg.norm(R0_bpms100, "fro")
norm_charge = np.linalg.norm(R2_bpms100-R0_bpms100, "fro") / np.linalg.norm(R0_bpms100, "fro")

print("Energy [%]: ", norm_energy * 100)
print("Charge [%]: ", norm_charge * 100)

Energy [%]:  82.23673131758625
Charge [%]:  1.0240912392086581e-05


### Quadrupoles misaligned by 0.02mm rms sigma.

In [15]:
directory_quads_misaligned_002_orbit = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/Quads_misaligned_0_02/SysID_kick_0_01/ATF2_Ext_RFT_quads_mis_0_02_sysidkick0_01_Orbit"
directory_quads_misaligned_002_energy = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/Quads_misaligned_0_02/SysID_kick_0_01/ATF2_Ext_RFT_quads_mis_0_02_sysidkick0_01_Dispersion"
directory_quads_misaligned_002_charge = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/Quads_misaligned_0_02/SysID_kick_0_01/ATF2_Ext_RFT_quads_mis_0_02_sysidkick0_01_Wakefield"

In [16]:
R0_response_quads2, R0_quads2 = _load_response(directory_quads_misaligned_002_orbit)
R1_response_quads2, R1_quads2 = _load_response(directory_quads_misaligned_002_energy)
R2_response_quads2, R2_quads2 = _load_response(directory_quads_misaligned_002_charge)

In [17]:
norm_energy = np.linalg.norm(R1_quads2-R0_quads2, "fro") / np.linalg.norm(R0_quads2, "fro")
norm_charge = np.linalg.norm(R2_quads2-R0_quads2, "fro") / np.linalg.norm(R0_quads2, "fro")

print("Energy [%]: ", norm_energy * 100)
print("Charge [%]: ", norm_charge * 100)

Energy [%]:  82.41042952695038
Charge [%]:  1.026528197562077e-05


### Quadrupoles misaligned by 0.100 rms sigma.

In [18]:
directory_quads_misaligned_0100_orbit = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/Quads_misaligned_0_100/ATF2_Ext_RFT_quads_mis_0_100_sysidkick0_01_Orbit"
directory_quads_misaligned_0100_energy = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/Quads_misaligned_0_100/ATF2_Ext_RFT_quads_mis_0_100_sysidkick0_01_Dispersion"
directory_quads_misaligned_0100_charge = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/Quads_misaligned_0_100/ATF2_Ext_RFT_quads_mis_0_100_sysidkick0_01_Wakefield"

In [19]:
R0_response_quads100, R0_quads100 = _load_response(directory_quads_misaligned_0100_orbit)
R1_response_quads100, R1_quads100 = _load_response(directory_quads_misaligned_0100_energy)
R2_response_quads100, R2_quads100 = _load_response(directory_quads_misaligned_0100_charge)

In [20]:
norm_energy = np.linalg.norm(R1_quads100-R0_quads100, "fro") / np.linalg.norm(R0_quads100, "fro")
norm_charge = np.linalg.norm(R2_quads100-R0_quads100, "fro") / np.linalg.norm(R0_quads100, "fro")

print("Energy [%]: ", norm_energy * 100)
print("Charge [%]: ", norm_charge * 100)

Energy [%]:  122.7266317344854
Charge [%]:  1.0587611511773632e-05


The bunch-charge dependence cannot currently be meaningfully evaluated because charge-dependent wakefield sources are not yet included in the simulation.

The norms above tell us how strongly an energy shift changes the
response matrix for each machine misalignment.

Which misalignments actually make response matrix not valid anymore?

In [21]:
bpm6_misalignment_effect = np.linalg.norm((R0_bpms6 - R0_aligned), "fro") / np.linalg.norm(R0_aligned, "fro") * 100
bpm100_misalignment_effect = np.linalg.norm((R0_bpms100 - R0_aligned), "fro") / np.linalg.norm(R0_aligned, "fro") * 100
quads2_misalignment_effect = np.linalg.norm((R0_quads2 - R0_aligned), "fro") / np.linalg.norm(R0_aligned, "fro") * 100
quads100_misalignment_effect = np.linalg.norm((R0_quads100 - R0_aligned), "fro") / np.linalg.norm(R0_aligned, "fro") * 100

print("BPM misaligned by 60um [%]:", bpm6_misalignment_effect)
print("BPM misaligned by 100um [%]:", bpm100_misalignment_effect)
print("Quads misaligned by 20um [%]:", quads2_misalignment_effect)
print("Quads misaligned by 100um [%]:", quads100_misalignment_effect)

BPM misaligned by 60um [%]: 8.299174080666461e-05
BPM misaligned by 100um [%]: 8.299174081217635e-05
Quads misaligned by 20um [%]: 2.8801611392196413
Quads misaligned by 100um [%]: 14.112178311328153


That is expected, because BPMs change response matrix the same way.

We measure response matrix for aligned machine again, to see if there is any noise floor.

In [22]:
directory_aligned_orbit_a = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/AlignedMachine/ATF2_Ext_RFT_evr_aligned_2_Orbit"
directory_aligned_orbit_b = "/Users/wiktoriamalek/CERN-Flight_Simulator-Data/BBA/ATF2_Ext_RFT/AlignedMachine/ATF2_Ext_RFT_evr_aligned_3_Orbit"
R0_response_aligned_a, R0_aligned_a = _load_response(directory_aligned_orbit_a)
R0_response_aligned_b, R0_aligned_b = _load_response(directory_aligned_orbit_b)

In [23]:
noise_floor = np.linalg.norm(R0_aligned_a - R0_aligned_b, "fro") / np.linalg.norm(0.5 * (R0_aligned_a + R0_aligned_b), "fro") * 100
print("Noise floor [%]:", noise_floor)

Noise floor [%]: 0.0


We simulate RLS.

In [24]:
R_initial = R0_aligned.copy()
R_shifted = R1_aligned.copy()

We want to to simulate corrector dithering.

In [25]:
rng = np.random.default_rng(55)
sysid_kick = 0.01
dither_amplitude = 0.1 * sysid_kick
gain = 0.4
corrector_limit = 0.05
rcond = 1e-3
T_energy_shift = 5

# Noise of one BPM reading, in the same position units as the orbit.
bpm_noise_std = 0.1 # 100 um
target_dither_snr = 5.0
p0_scale = 1.0  # Initial uncertainty in normalized-dither units.

def generate_kick(R_hat, x, iteration):
    # correction using current, not ideal A
    n_corrs = R_hat.shape[1]
    u_correction = -gain * (np.linalg.pinv(R_hat, rcond = rcond) @ x)
    correction_limit = corrector_limit - dither_amplitude
    u_correction = np.clip(u_correction, -correction_limit, correction_limit)
    j = iteration % n_corrs # we choose correctors one after another for dithering
    dither = np.zeros(n_corrs)
    dither[j] = dither_amplitude * rng.choice([-1.0, 1.0]) # random sign (plus or minus)
    return u_correction + dither

def simulate_response(iteration, u_t):
    if iteration < T_energy_shift:
        R_true = R_initial
    else:
        R_true = R_shifted
    delta_x_true = R_true @ u_t
    noise_on_mean = bpm_noise_std / np.sqrt(samples_per_measurement)
    delta_x_observed = delta_x_true + rng.normal(0.0, noise_on_mean, size=R_true.shape[0])
    return delta_x_observed, delta_x_true, R_true

RLS updates the response matrix based on a single observation:

$$
u_t \longrightarrow \Delta x^{\mathrm{observed}}_t
$$

The algorithm first predicts the BPM change:

$$
\Delta x^{\mathrm{predicted}}_t = \hat{R}_t u_t
$$

It then calculates the prediction error:

$$
e_t = \Delta x^{\mathrm{observed}}_t - \hat{R}_t u_t
$$

In [26]:
def rls_update(R_hat, P, u_t, dx_observed, forgetting_factor):
    dx_predicted = R_hat @ u_t # how well current R predicts observation
    e_t = dx_observed - dx_predicted # size of n_bpms, u_t has size of n_corr, so one observation can update the full matrix
    P_new = (1/forgetting_factor) * (P - np.outer(P @ u_t, u_t @ P) / (forgetting_factor + u_t @ P @ u_t)) # how much RLS is not sure about its prediction, the bigger, the more changes RLS applies to R
    R_hat_new = R_hat + np.outer(e_t, u_t @ P) / (forgetting_factor + u_t @ P @ u_t) # if the matrices are nearly the same, e_t = 0 and the other part of the equation is 0
    return R_hat_new, P_new

In [27]:
n_corrs = R_initial.shape[1]
n_iterations = 10 * n_corrs
T_energy_shift = 3 * n_corrs
forgetting = 0.98

dither_signal_rms = np.median([np.sqrt(np.mean((R_initial[:, j] * dither_amplitude) ** 2)) for j in range(n_corrs)])
#samples_per_measurement = max(1, int(np.ceil((target_dither_snr * bpm_noise_std / dither_signal_rms) ** 2)))
samples_per_measurement = 3
single_shot_snr = dither_signal_rms / bpm_noise_std
effective_snr = dither_signal_rms / (bpm_noise_std / np.sqrt(samples_per_measurement))
print(f'Dither signal RMS: {dither_signal_rms:.3g}')
print(f'Single-shot dither SNR: {single_shot_snr:.3g}')
print(f'Averaging {samples_per_measurement} BPM readings per update gives SNR: {effective_snr:.3g}')

# Normalize corrector kicks so that a dither has magnitude one.
# RLS is scale-sensitive: with raw 1e-3 kicks, P0 = I would make its gain almost zero.
u_scale = dither_amplitude
Theta_hat = R_initial * u_scale  # Theta maps normalized kicks to BPM displacement.

P = p0_scale * np.eye(n_corrs)

initial_error_kicks = rng.uniform(-0.5*corrector_limit, 0.5*corrector_limit, (n_corrs,))
x = R_initial @ initial_error_kicks
model_error = []
static_model_error = []
orbit_residual = []
applied_kicks = []
for i in range(n_iterations):

    R_hat = Theta_hat / u_scale
    u_t = generate_kick(R_hat, x, i) # corrector kicks

    delta_x_observed, delta_x_true, R_true = simulate_response(i, u_t)

    u_normalized = u_t / u_scale
    Theta_hat, P = rls_update(Theta_hat, P, u_normalized, delta_x_observed, forgetting)

    x = x + delta_x_true  # Measurement noise must not be added to the physical orbit.

    R_hat = Theta_hat / u_scale
    adaptive_matrix_error = (np.linalg.norm(R_hat - R_true, "fro") / np.linalg.norm(R_true, "fro"))
    static_matrix_error = (np.linalg.norm(R_initial - R_true, "fro") / np.linalg.norm(R_true, "fro"))

    model_error.append(100 * adaptive_matrix_error)
    static_model_error.append(100 * static_matrix_error)
    orbit_residual.append(np.linalg.norm(x))
    applied_kicks.append(u_t.copy())

Dither signal RMS: 0.000299
Single-shot dither SNR: 0.00299
Averaging 3 BPM readings per update gives SNR: 0.00518


In [28]:
plt.figure(figsize=(8, 4))

plt.semilogy(model_error, label="Adaptive RLS")
plt.semilogy(static_model_error, "--", label="Static $R_{initial}$")

plt.axvline(T_energy_shift, color="black", linestyle=":", label="Energy shift")
plt.xlabel("Iteration")
plt.ylabel(r"Relative response-matrix error [%]")
plt.title("Adapting the response matrix after energy shift")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.show()

Can RLS reproduce the known response matrix with both plus and minus files?

In [29]:
pairs = list_pair_files(directory_aligned_orbit)
print("Number of pairs:", len(pairs))
hcorrs = R0_response_aligned.hcorrs
vcorrs = R0_response_aligned.vcorrs
bpms = R0_response_aligned.bpms
def extract_step(pkl_path):
    S = State(filename=pkl_path)
    u_h = np.asarray(S.get_correctors(hcorrs)["bact"], dtype=float)
    u_v = np.asarray(S.get_correctors(vcorrs)["bact"], dtype=float)
    u_abs = np.concatenate([u_h, u_v])
    orbit = S.get_orbit(bpms)
    x_abs = np.concatenate([
        np.asarray(orbit["x"], dtype=float),
        np.asarray(orbit["y"], dtype=float)
    ])
    return u_abs, x_abs

Number of pairs: 58


In [30]:
# R_hat_files = np.zeros_like(R0_aligned)
# P_files = (1 / sysid_kick**2) * np.eye(R_hat_files.shape[1])
# error_norms = []
# for file_plus, file_minus, tag in pairs:
#     u_plus, x_plus = extract_step(file_plus)
#     u_minus, x_minus = extract_step(file_minus)
#     delta_u = u_plus - u_minus
#     delta_x = x_plus - x_minus
#     R_hat_files, P_files = rls_update(R_hat_files, P_files, delta_u, delta_x, forgetting_factor=1.0)
#     error_norm = (np.linalg.norm(R_hat_files - R0_aligned, "fro") / np.linalg.norm(R0_aligned, "fro"))
#     error_norms.append(100 * error_norm)
#
# plt.figure(figsize=(8, 4))
# plt.plot(error_norms)
# plt.xlabel("Number of processed p/m pairs")
# plt.ylabel("Relative error vs $R_{0,aligned}$ [%]")
# plt.title("Can RLS reproduce the known response matrix?")
# plt.grid(alpha=0.3)
# plt.show()

We want to see if RLS converges better with plus and minus files or if only one is sufficient.

Now we check the experiment with only 'plus' files.

In [31]:
# plus_files = list_plus_files(directory_aligned_orbit)
#
# R_hat_single = np.zeros_like(R0_aligned)
# P_single = ((1 / sysid_kick**2) * np.eye(R_hat_single.shape[1]))
# single_error_norms = []
#
# for i in range(len(plus_files) - 1):
#     u_prev, x_prev = extract_step(plus_files[i])
#     u_curr, x_curr = extract_step(plus_files[i + 1])
#     delta_u = u_curr - u_prev
#     delta_x = x_curr - x_prev
#     R_hat_single, P_single = rls_update(R_hat_single, P_single, delta_u, delta_x, forgetting_factor=1.0)
#     error_norm = (np.linalg.norm(R_hat_single - R0_aligned, "fro") / np.linalg.norm(R0_aligned, "fro"))
#     single_error_norms.append(100 * error_norm)
#
# plt.figure(figsize=(8, 4))
# plt.plot(single_error_norms, label="Only + files")
# plt.xlabel("Number of processed differences")
# plt.ylabel("Relative error vs $R_{0,aligned}$ [%]")
# plt.title("Can RLS reproduce R using only single files?")
# plt.grid(alpha=0.3)
# plt.legend()
# plt.show()

In [32]:
# plt.figure(figsize=(8, 4))
#
# plt.plot(error_norms, label="p/m pairs")
# plt.plot(single_error_norms, label="single + files")
#
# plt.xlabel("Number of processed samples")
# plt.ylabel(r"Relative error vs $R_{0,\mathrm{aligned}}$ [%]")
# plt.title("Response-matrix reconstruction with RLS")
# plt.grid(alpha=0.3)
# plt.legend()
# plt.show()

### RLS reconstruction with misaligned quadrupoles

Starting from the aligned response matrix, reconstruct the response measured with 20 micrometre rms quadrupole misalignments. Each `+/-` pair supplies one differential observation to RLS.

In [33]:
# misaligned_directory = directory_quads_misaligned_002_orbit
# R_true_misaligned = R0_quads2
# misaligned_pairs = list_pair_files(misaligned_directory)
#
# R_hat_misaligned = R0_aligned.copy()
# P_misaligned = (1 / sysid_kick**2) * np.eye(R_hat_misaligned.shape[1])
# initial_error = 100 * np.linalg.norm(R_hat_misaligned - R_true_misaligned, 'fro') / np.linalg.norm(R_true_misaligned, 'fro')
# misaligned_error_norms = []
#
# for file_plus, file_minus, _tag in misaligned_pairs:
#     u_plus, x_plus = extract_step(file_plus)
#     u_minus, x_minus = extract_step(file_minus)
#     delta_u = u_plus - u_minus
#     delta_x = x_plus - x_minus
#     R_hat_misaligned, P_misaligned = rls_update(
#         R_hat_misaligned, P_misaligned, delta_u, delta_x, forgetting_factor=1.0
#     )
#     error = np.linalg.norm(R_hat_misaligned - R_true_misaligned, 'fro') / np.linalg.norm(R_true_misaligned, 'fro')
#     misaligned_error_norms.append(100 * error)
#
# plt.figure(figsize=(8, 4))
# plt.plot(misaligned_error_norms, label='RLS estimate')
# plt.axhline(initial_error, color='tab:orange', linestyle='--', label='Aligned matrix (no update)')
# plt.xlabel('Number of processed p/m pairs')
# plt.ylabel('Relative error vs misaligned response matrix [%]')
# plt.title('RLS reconstruction after quadrupole misalignment')
# plt.grid(alpha=0.3)
# plt.legend()
# plt.show()

### Fixed misaligned-quadrupole recovery with BPM noise

This benchmark keeps the true response matrix fixed after a 20 micrometre rms quadrupole misalignment. It isolates RLS matrix recovery from BPM noise; it is not a drift-tracking test.

In [34]:
R_true_fixed = R0_quads2
R_hat_fixed = R0_aligned.copy()

fixed_bpm_noise_std = 0.1
fixed_samples_per_setting = 3
fixed_dither_amplitude = 0.001
fixed_n_cycles = 10

n_correctors = R_true_fixed.shape[1]
pair_kick = 2 * fixed_dither_amplitude
Theta_fixed = R_hat_fixed * pair_kick
P_fixed = np.eye(n_correctors)
fixed_errors = []
initial_fixed_error = 100 * np.linalg.norm(R_hat_fixed - R_true_fixed, 'fro') / np.linalg.norm(R_true_fixed, 'fro')

pair_noise_std = fixed_bpm_noise_std * np.sqrt(2 / fixed_samples_per_setting)
pair_signal_rms = np.median([
    np.sqrt(np.mean((R_true_fixed[:, j] * pair_kick) ** 2))
    for j in range(n_correctors)
])
print(f'Pair SNR at one corrector: {pair_signal_rms / pair_noise_std:.3g}')

for step in range(fixed_n_cycles * n_correctors):
    j = step % n_correctors
    delta_u = np.zeros(n_correctors)
    delta_u[j] = pair_kick
    delta_x_true = R_true_fixed @ delta_u
    delta_x_observed = delta_x_true + rng.normal(0.0, pair_noise_std, size=R_true_fixed.shape[0])

    Theta_fixed, P_fixed = rls_update(Theta_fixed, P_fixed, delta_u / pair_kick, delta_x_observed, forgetting_factor=1.0)
    R_hat_fixed = Theta_fixed / pair_kick
    fixed_errors.append(100 * np.linalg.norm(R_hat_fixed - R_true_fixed, 'fro') / np.linalg.norm(R_true_fixed, 'fro'))

plt.figure(figsize=(8, 4))
plt.semilogy(fixed_errors, label='RLS estimation')
plt.axhline(initial_fixed_error, color='tab:orange', linestyle='--', label='Aligned matrix (no update)')
plt.xlabel('Processed +/- pairs')
plt.ylabel('Relative error vs fixed misaligned matrix [%]')
plt.title('RLS update with BPM noise and quadrupole misalignment')
plt.grid(alpha=0.3)
plt.legend()
plt.show()

Pair SNR at one corrector: 0.0073
